<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

## 🎯 CrewAI Hello World — Lesson Notes

**One-liner**: CrewAI lets you orchestrate multiple LLM-powered `Agent`s, each with a role/goal/backstory, to jointly complete a set of `Task`s inside a `Crew`.

</div>

<div class="dark-info" style="background:#1e293b; color:#e2e8f0; padding:12px 24px; border-left:4px solid #60a5fa; border-radius:4px; width:97%;"><style>.dark-info strong{color:#fbbf24;}</style>

**What problem does CrewAI solve?**

Instead of writing one giant prompt and hoping an LLM does everything correctly, CrewAI lets you decompose a job into **role-playing agents** (each with a narrow specialty) and **tasks** (each with a clear description + expected output). A `Crew` wires agents and tasks together and runs them, so the LLM's reasoning is scoped and structured instead of open-ended.

In this lesson's example: a single **Content Planning Agent** plans a week of YouTube Shorts about the surprising origins of everyday things.

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:16px 28px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🧱 Core Building Blocks

</div>

| Class | Key params | What it represents |
|---|---|---|
| **`Agent`** | `role`, `goal`, `backstory`, `llm`, `verbose` | A team member: *who* they are (`role`/`backstory`) and *what* they're trying to achieve (`goal`). `backstory` isn't decoration — it's prompt context that steers how the LLM roleplays. |
| **`Task`** | `description`, `expected_output`, `agent` | A concrete unit of work: *what* to do (`description`) and *what shape* the result must have (`expected_output`, e.g. a JSON schema). `agent=` binds it to exactly one executor. |
| **`Crew`** | `agents`, `tasks`, `process` | The orchestrator: holds the roster (`agents`) and the work order (`tasks`), then runs them according to `process` (default: `Process.sequential`). |

<div class="dark-orange" style="background:#2d2418; color:#fed7aa; padding:10px 24px; border-left:4px solid #fb923c; border-radius:4px; width:97%;"><style>.dark-orange strong{color:#67e8f9;}</style>

💡 **Mental model**: `Agent` = a persona + LLM. `Task` = an assignment with a done-criteria. `Crew` = the scheduler that decides who runs what, in what order, and stitches results together.

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:16px 28px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🔬 Walkthrough of This Lesson's Example

</div>

<pre style="background:#1e1e2e; color:#d4d4d4; padding:10px 24px; border-radius:4px; font-size:0.90em; width:97%;">
<code class="language-python">
content_creator_assistant = Agent(
    role="YouTube Shorts Micro-History Strategist",
    goal="Plan a 1-week slate of high-retention YouTube Shorts...",
    backstory="You specialize in 30-45s micro-history that hooks fast...",
    llm="gpt-4o-mini",
    verbose=True,
)

task = Task(
    description="Create a 1-week video posting plan with 5 video blueprints...",
    expected_output=<span style="background:#3d3414; color:#fde047; padding:0 2px;">'''JSON schema with a "videos" array...'''</span>,
    agent=content_creator_assistant,
)

crew = Crew(agents=[content_creator_assistant], tasks=[task])
result = crew.kickoff()
</code></pre>

One agent, one task: the crew executes the task once, the agent generates a JSON array of 5 video blueprints (title, hook, visuals, tags, CTA), and `result.raw` holds that JSON as text.

<div class="dark-orange" style="background:#2d2418; color:#fed7aa; padding:10px 24px; border-left:4px solid #fb923c; border-radius:4px; width:97%;"><style>.dark-orange strong{color:#67e8f9;}</style>

💡 **Setup notes worth remembering**

- `os.environ["OPENAI_API_KEY"] = get_openai_api_key()` — `utils.py` calls `load_dotenv(find_dotenv())` under the hood, so a `.env` file with `OPENAI_API_KEY=...` must sit somewhere above (or in) the working directory.
- Dependencies are installed via **uv**, using a `pyproject.toml` listing `crewai`, `python-dotenv`, `ipykernel` — run `uv sync` then `uv run jupyter lab ...` inside the lesson folder.

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:16px 28px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### ❓ Common Confusion — `agents=[...]` vs `tasks=[...]`

</div>

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**Key insight**: the `agents=[...]` list on a `Crew` does **not** decide execution order. Order and "who does what" come entirely from the **`tasks=[...]` list order** plus each `Task`'s own `agent=` field.

</div>

**`Process.sequential` (the default) — what actually happens:**

| Step | Behavior |
|---|---|
| 1 | Crew iterates `tasks` strictly in list order — `tasks[0]`, `tasks[1]`, ... |
| 2 | Each task is executed by **its own** `agent=` binding, not by "whichever agent is free" |
| 3 | Task `i`'s output is auto-injected as context into task `i+1`, unless that task sets its own `context=[...]` |
| 4 | `crew.kickoff()` returns a `CrewOutput` — its `.raw` is the **last** task's output |
| 5 | `result.tasks_output` holds a list of *every* intermediate task's output, in order |

<div class="dark-info" style="background:#1e293b; color:#e2e8f0; padding:12px 24px; border-left:4px solid #60a5fa; border-radius:4px; width:97%;"><style>.dark-info strong{color:#fbbf24;}</style>

**So what is `agents=[...]` actually for?**

In `Process.sequential`, it's mostly a **declared roster** — a fallback the Crew can draw from if a `Task` has no explicit `agent=` (best practice: always set `agent=` explicitly to avoid ambiguity).

It becomes genuinely load-bearing in **`Process.hierarchical`**: CrewAI spins up an implicit **manager agent** that treats `agents=[...]` as a pool of workers and *dynamically* assigns each task to whichever agent fits — execution order is then decided by the manager's judgment, not strictly by the `tasks` list.

</div>

**Extending beyond "hello world" — a multi-agent, multi-task crew:**

<pre style="background:#1e1e2e; color:#d4d4d4; padding:10px 24px; border-radius:4px; font-size:0.90em; width:97%;">
<code class="language-python">
crew = Crew(
    agents=[researcher, planner, writer],
    tasks=[research_task, plan_task, write_task],
    process=Process.sequential,
)
# research_task.agent = researcher
# plan_task.agent    = planner
# write_task.agent   = writer

result = crew.kickoff()
<span style="background:#3d3414; color:#fde047; padding:0 2px;"># result.raw            -> write_task's output only
# result.tasks_output    -> [research_task_out, plan_task_out, write_task_out]</span>
</code></pre>

This lesson's crew is the degenerate case of the pattern above: 1 agent, 1 task, so `result.raw` *is* the only task's output — that's precisely why it's called "hello world".

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:16px 28px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🐛 Runtime Pitfall Hit While Running This Lesson

</div>

<div class="dark-error" style="background:#2d1f1f; color:#fca5a5; padding:10px 24px; border-left:4px solid #f87171; border-radius:4px; width:97%;"><style>.dark-error strong{color:#fde047;}</style>

Running `result = crew.kickoff()` inside Jupyter raised:

```
RuntimeError: Agent execution was invoked synchronously from within a
running event loop. Use `agent.kickoff_async()` / `crew.kickoff_async()`
(or `await agent.aexecute_task(...)`) when calling from async code.
```

**Why**: Jupyter's `ipykernel` already runs its own asyncio event loop under the hood. This version of `crewai` executes agents via an async path internally, and its sync `kickoff()` refuses to run when it detects it's already inside a running event loop — a conflict that only shows up in notebooks (a plain `.py` script has no pre-existing loop, so `crew.kickoff()` works fine there).

</div>

<div class="dark-orange" style="background:#2d2418; color:#fed7aa; padding:10px 24px; border-left:4px solid #fb923c; border-radius:4px; width:97%;"><style>.dark-orange strong{color:#67e8f9;}</style>

💡 **The fix — use the async kickoff in notebooks:**

</div>

<pre style="background:#1e1e2e; color:#d4d4d4; padding:10px 24px; border-radius:4px; font-size:0.90em; width:97%;">
<code class="language-python">
result = <span style="background:#3d3414; color:#fde047; padding:0 2px;">await crew.kickoff_async()</span>
</code></pre>

Jupyter supports top-level `await` in a cell, so this is a one-line fix. Alternative, if you want to keep the synchronous `crew.kickoff()` call: patch the event loop with `nest_asyncio`.

<pre style="background:#1e1e2e; color:#d4d4d4; padding:10px 24px; border-radius:4px; font-size:0.90em; width:97%;">
<code class="language-python">
import nest_asyncio
nest_asyncio.apply()

result = crew.kickoff()  # now safe to call synchronously
</code></pre>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:16px 28px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### ✨ Key Takeaways

</div>

<div class="dark-success" style="background:#1a2e1f; color:#bbf7d0; padding:10px 24px; border-left:4px solid #4ade80; border-radius:4px; width:97%;"><style>.dark-success strong{color:#fbbf24;}</style>

- `Agent` = persona + LLM. `Task` = assignment + done-criteria. `Crew` = scheduler.
- **Order and ownership come from `tasks` (+ each task's `agent=`), not from `agents`.**
- `Process.sequential` chains tasks, auto-passing context forward; `result.raw` is the last task's output, `result.tasks_output` is all of them.
- `agents=[...]` only becomes the real dispatcher in `Process.hierarchical`, via an implicit manager agent.
- In Jupyter, prefer `await crew.kickoff_async()` over `crew.kickoff()` to avoid the nested-event-loop `RuntimeError`.

</div>